# M8a K-means 군집화 — 실습 (W12, M8 2부작 1편)

> ⚠️ **가장 먼저 — 화면 위 [Drive로 복사]를 누르세요.**
> 지금 보고 있는 것은 원본을 잠깐 띄운 **임시 사본**입니다. 복사하지 않으면 탭을 닫는 순간
> 채운 빈칸과 실행 결과가 **모두 사라집니다.** 복사본은 내 Google Drive에 저장되고,
> 원본은 바뀌지 않으니 마음껏 고쳐도 됩니다.

> 위에서부터 한 셀씩 `Shift+Enter`로 실행하세요. `___` 빈칸은 직접 채웁니다.

**이 실습이 끝나면**
1. 1차원 7점의 **K-means 손계산 궤적**([1.5, 8] → [2.5, 11] 수렴, 관성 7.0)을 코드로 재현하고 **sklearn 검산**으로 일치를 확인한다 ⭐
2. blobs 군집·**엘보우**(꺾임 k=4)·**초기화 운**(22.8배)을 실측으로 본다
3. **스케일링 실험**(0.517 → 1.000)과 **초승달 실패**(0.75)로 K-means의 조건과 한계를 확인한다

**7단계 멘탈모델 초점:** 표현(비지도) — 첫 비지도학습!

## Part A. 세트피스 — 세상에서 가장 작은 K-means ⭐
점 {1, 2, 3, 4, 10, 11, 12}, k=2, **일부러 나쁜 시작**(중심 1과 4). 규칙 두 개 — ①할당(가까운 중심으로) ②갱신(군집의 평균으로) — 를 코드로 돌리며, 먼저 종이에 예상 궤적을 적어 보세요.

In [ ]:
import numpy as np                                     # 수치 계산

pts = np.array([1, 2, 3, 4, 10, 11, 12], dtype=float)  # 점 7개(누가 봐도 두 덩어리)
c = np.array([1.0, 4.0])                               # 나쁜 시작: 둘 다 왼쪽 덩어리 안!
for it in range(1, 6):                                 # 수렴까지 반복
    dist = np.abs(pts[:, None] - c[None, :])           # 각 점 → 두 중심의 거리 (7, 2)
    assign = np.argmin(dist, axis=___)                 # ✍️ 빈칸: ①할당 — 어느 축 방향의 최솟값? (중심 축)
    new_c = np.array([pts[assign == j].___() for j in range(2)])  # ✍️ 빈칸: ②갱신 — 군집의 무엇으로?
    print(f'반복{it}: 할당={assign.tolist()} → 새 중심={new_c.tolist()}')
    if np.allclose(new_c, c):                          # 중심이 안 움직이면
        print('수렴!')                                  # 종료
        break
    c = new_c                                          # 다음 반복으로

inertia = sum(((pts[assign == j] - c[j]) ** 2).sum() for j in range(2))  # 관성 = 거리² 합
print('최종 중심:', c.tolist(), '| 관성:', inertia)      # [2.5, 11.0] / 7.0 — 손계산과 같은가?

In [ ]:
from sklearn.cluster import KMeans                     # 이제 sklearn으로 검산

km = KMeans(n_clusters=2, init=np.array([[1.0], [4.0]]), n_init=1)  # 같은 나쁜 시작을 지정
km.fit(pts.reshape(-1, 1))                             # 1차원 → (7, 1) 형태로 학습
print('sklearn 중심:', sorted(km.cluster_centers_.ravel().tolist()))  # [2.5, 11.0]?
print('sklearn 관성:', round(km.inertia_, 4))           # 7.0?

km1 = KMeans(n_clusters=1, n_init=10, random_state=0).fit(pts.reshape(-1, 1))  # 비교: 덩어리 하나로 뭉개면
print('k=1 관성:', round(km1.inertia_, 2))              # 130.86 — 나눠 담는 이득!

> **검산 포인트:** 궤적 — 반복 1에서 3은 c₂(4)로 갔다가([0,0,1,1,1,1,1] → 중심 [1.5, 8]), 반복 2에서 **3과 4가 c₁로 이적**([0,0,0,0,1,1,1] → [2.5, 11]), 반복 3에서 수렴. sklearn도 최종 중심 **[2.5, 11] · 관성 7.0** — 일치. 엔진은 **"평균이 중심을 데이터 쪽으로 끌고 간다"**(c₂가 오른쪽 덩어리를 끌어안자 평균이 8로 끌려가고, 그 덕에 3·4가 넘어올 길이 열림).

## Part B. 실전 — blobs 300점 군집
2차원 인공 데이터 4덩어리를 K-means로 묶고 중심을 그려 봅니다. **정답 라벨은 일부러 버립니다**(`_`) — 비지도니까!

In [ ]:
import matplotlib.pyplot as plt                        # 그래프
from sklearn.datasets import make_blobs                # 인공 군집 데이터

X, _ = make_blobs(n_samples=300, centers=4,            # 4덩어리 300점 — 정답 라벨은 버림(비지도!)
                  cluster_std=0.8, random_state=42)

km4 = KMeans(n_clusters=4, random_state=42, n_init=10)  # k=4, 보험(n_init) 포함
labels = km4.fit_predict(X)                            # 학습 + 각 점의 군집 번호
centers = km4.___                                      # ✍️ 빈칸: 학습된 군집 중심 좌표 속성

plt.scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis', s=20)          # 군집별 색
plt.scatter(centers[:, 0], centers[:, 1], c='red', marker='X', s=200, edgecolor='k')  # 중심
plt.title('K-means clustering (k=4)')                  # 제목(영어)
plt.show()
print('관성:', round(km4.inertia_, 1))                  # 362.5

> **관찰:** 정답을 전혀 주지 않았는데 4덩어리로 깔끔하게 묶이고, 빨간 X(중심)가 각 덩어리의 평균에 자리 잡았습니다 — Part A의 두 규칙이 2차원에서도 그대로 작동한 것.

## Part C. 엘보우 — k는 사람이 정한다
k를 1→8로 돌리며 관성을 기록합니다. **꺾임**을 찾으세요 — 최솟값이 아니라!

In [ ]:
inertias = []                                          # 관성 저장
ks = range(1, 9)                                       # k = 1~8
for k in ks:                                           # 각 k에 대해
    m = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)  # 학습
    inertias.append(m.___)                             # ✍️ 빈칸: 관성(점→중심 거리² 합) 속성

plt.plot(list(ks), inertias, 'o-')                     # 관성 곡선
plt.xlabel('k (number of clusters)'); plt.ylabel('inertia')  # 축(영어)
plt.grid(True, alpha=0.3)
plt.title('Elbow method')
plt.show()
print('관성:', [round(v) for v in inertias])            # 3→4에서 급감, 이후 찔끔찔끔

> **관찰:** [19780, 9211, **1919 → 362**, 329, 295, 262, 232] — k=3→4에서 급감 후 평평 = **꺾임(elbow) = k=4**(실제 4덩어리와 일치). ⚠️ 최솟값으로 고르면 안 되는 이유: k↑면 관성은 **항상**↓(점마다 군집이면 0 — 과적합의 비지도판). M2a의 "train 최고점을 고르지 않는다"와 같은 리듬.

## Part D. 스케일링 실험 — M3의 교훈 그대로 ⭐
정보는 **특징 A(범위 ~1)** 에만 있고, **특징 B는 정보 없이 범위만 큰(0~100)** 데이터를 만들어 — 원본 vs 표준화 후의 군집을 비교합니다.

In [ ]:
from sklearn.preprocessing import StandardScaler       # 표준화(M3에서 배운 그 도구)

rng = np.random.default_rng(0)                         # 재현성
n = 150                                                # 그룹당 150점
xa, xb = rng.normal(0.0, 0.15, n), rng.normal(1.0, 0.15, n)   # 특징 A: 두 그룹이 0과 1 근처로 갈림
ya, yb = rng.uniform(0, 100, n), rng.uniform(0, 100, n)       # 특징 B: 정보 없음, 범위만 0~100
Xg = np.vstack([np.column_stack([xa, ya]), np.column_stack([xb, yb])])  # (300, 2)
g_true = np.array([0] * n + [1] * n)                   # 진짜 그룹(채점용)

def match_rate(labels, truth):                         # 군집→다수 그룹 매핑 일치율
    total = 0
    for j in np.unique(labels):                        # 각 군집을
        _, cnts = np.unique(truth[labels == j], return_counts=True)  # 다수 그룹에 매핑
        total += cnts.max()
    return total / len(truth)

lab_raw = KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(Xg)   # 원본 그대로
Xg_s = StandardScaler().___(Xg)                        # ✍️ 빈칸: 표준화 학습+변환을 한 번에 하는 메서드
lab_sc = KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(Xg_s)  # 표준화 후
print('원본 일치율   :', round(match_rate(lab_raw, g_true), 3))   # 0.517 — 동전 던지기!
print('표준화 일치율 :', round(match_rate(lab_sc, g_true), 3))    # 1.0

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))      # 두 결과를 나란히
axes[0].scatter(Xg[:, 0], Xg[:, 1], c=lab_raw, cmap='coolwarm', s=15)
axes[0].set_title('raw: distance dominated by feature B')
axes[1].scatter(Xg[:, 0], Xg[:, 1], c=lab_sc, cmap='coolwarm', s=15)
axes[1].set_title('scaled: groups recovered')
for ax in axes:
    ax.set_xlabel('feature A (informative)'); ax.set_ylabel('feature B (large scale)')
plt.tight_layout(); plt.show()

> **관찰:** 원본은 **0.517(동전)** — 거리가 특징 B에 끌려가 **엉뚱한 방향(위/아래)으로 반 토막**. 표준화 후 **1.000**. KNN(M3)과 K-means는 정답의 유무만 다를 뿐 **같은 배**(거리 기반 = 스케일링 필수). M6의 트리에게는 없던 걱정 — 모델마다 체질이 다릅니다.

## Part E. 초기화의 운 + 초승달 실패
무작위 초기화 1회의 운(지역 최적에 갇히기)과, 둥근 덩어리 가정이 깨지는 데이터(make_moons)를 확인합니다.

In [ ]:
best = KMeans(n_clusters=4, random_state=42, n_init=___).fit(X)  # ✍️ 빈칸: 보험 — 이 실습의 재시작 횟수(예전 sklearn 기본값이던 그 수)
print('k-means++ 재시작 10회:', round(best.inertia_, 1))          # 362.5
for seed in (1, 4):                                    # 완전 무작위 1회 — 시드 운 비교
    bad = KMeans(n_clusters=4, init='random', n_init=1, random_state=seed).fit(X)
    print(f'무작위 1회 (seed {seed}):', round(bad.inertia_, 1))    # 8255.8 vs 362.5 — 22.8배!

from sklearn.datasets import make_moons                # 초승달 데이터
Xm, ym = make_moons(n_samples=300, noise=0.06, random_state=42)  # 두 달이 얽힌 모양
labm = KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(Xm)  # k=2로 시도
print('초승달 일치율:', round(match_rate(labm, ym), 3))  # 0.75 — 허리를 뚝!

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(Xm[:, 0], Xm[:, 1], c=ym, cmap='coolwarm', s=15)     # 진짜 모양
axes[0].set_title('true shape: two moons')
axes[1].scatter(Xm[:, 0], Xm[:, 1], c=labm, cmap='coolwarm', s=15)   # K-means 결과
axes[1].set_title('K-means (k=2): cuts across the moons')
plt.tight_layout(); plt.show()

> **관찰:** ① 무작위 1회는 시드에 따라 362.5(운)  vs **8255.8(22.8배 — 갇힘)**. 보험 = **k-means++**(sklearn 기본값) + **n_init=10 명시**(⚠️ 현 sklearn 기본은 `'auto'`=k-means++일 때 1회 — 직접 지정하는 습관). ② 초승달 일치율 **0.75** — "중심에서 가까운 점"이라는 정의로는 초승달을 표현할 수 없어 허리를 직선으로 자릅니다(🔹심화: DBSCAN). **다음 주(M8b):** 묶었으니 **줄일** 차례 — 64차원 손글씨를 2D로, "같은 데이터, 축만 바꿨는데 50:50이 80:20".

## 🤖 AI 코파일럿 활용 (선택) — ai-native v1
막히면 AI 튜터에게 묻되, **먼저 스스로 생각**하고 답을 **실행으로 검증**하세요.

**좋은 질문 예시**
- "점 {2, 4, 20, 22}, k=2, 시작 중심 2와 4 — 내가 손으로 돌려 볼 테니 채점해 줘."
- "관성 최솟값으로 k를 고르면 왜 안 되는지 설명해 볼게 — 허점을 찔러 줘."
- "K-means와 KNN의 공통점(거리·스케일링)과 차이(정답 유무)를 말해 볼게."
- "초승달에서 K-means가 실패하는 이유를 '중심'으로 설명해 볼게 — DBSCAN은 왜 되는지도 물어볼게."

**가드레일**
1. 먼저 손으로 생각 → 그 다음 AI
2. AI 코드는 *왜 그런지* 설명할 수 있을 때만 사용
3. AI 출력은 실행으로 검증

## 정리 & 자가 점검

**오늘 한 일 3줄**
1. 7점 K-means를 **수렴까지 손계산**([1.5, 8] → [2.5, 11], 관성 7.0)하고 sklearn 검산 일치를 확인했다 — 첫 비지도학습!
2. blobs 군집·엘보우(꺾임 k=4, 최솟값 함정)·초기화 운(22.8배, 보험 2종)을 실측으로 봤다
3. 스케일링 실험(0.517 → 1.000 — M3와 같은 배)과 초승달 실패(0.75 — 둥근 덩어리 가정)를 확인했다

**스스로 점검**
- [ ] 할당·갱신 두 규칙을 종이에 재현할 수 있다(이적 포함)
- [ ] 관성의 정의와 "최솟값 함정"을 설명할 수 있다
- [ ] 초기화가 왜 문제이고 무엇이 보험인지 안다
- [ ] K-means에 스케일링이 필수인 이유를 M3와 연결할 수 있다
- [ ] K-means가 실패하는 모양을 하나 들 수 있다

**🔹심화 (선택)**
- `silhouette_score`로 k=2~8의 실루엣을 재 보세요 — 엘보우와 같은 k를 가리키나요?
- Part E의 초승달에 `DBSCAN(eps=0.2)`을 적용해 보세요 — 일치율이 어떻게 되나요?
- Part A의 시작 중심을 [10, 11]로 바꿔 보세요 — 이번에도 올바른 두 덩어리를 찾나요? 몇 번 만에?

**다음 시간(M8b):** PCA — 분산 몰아주기 손계산 · 64차원 → 2D · 설명된 분산.